provided by cbi  
https://github.com/cbi-society/cheminfo_tutorial_20241028_pub/tree/main

subprocess.callで実行する場合、存在するtomlファイルを指定すればreinventは実行できる。
1. tomlファイルをsubprocess.callで編集できれば実行できる
2. tomlファイルをつくらずに実行できればなおよい

# 変数処理のテスト

In [ ]:
import subprocess

In [2]:
test='''
this is a test
hello world from python and terminal
'''

In [3]:
subprocess.call([
    'echo',
     test
     ])

In [50]:
random='''
run_type = "sampling"
device = "mps"  # set torch device e.g. "cpu". For macOS, use "mps"
json_out_config = "_sampling.json"  # write this TOML to JSON

## Reinvent: de novo sampling
model_file = "User/keetane/Documents/apps/REINVENT4/priors/reinvent.prior"
output_file = 'sampling.csv'  # sampled SMILES and NLL in CSV format

num_smiles = 157  # number of SMILES to be sampled, 1 per input SMILES
unique_molecules = true  # if true remove all duplicatesd canonicalize smiles
randomize_smiles = true # if true shuffle atoms in SMILES randomly

'''
subprocess.call(['cat', random])


In [51]:
subprocess.call(['echo', random])

# REINVENT4の実行

## Working Directoryとpathの設定

temporary directoryをworking directoryに設定する。

import libraries

In [ ]:
import os
import shutil

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import reinvent
from reinvent.notebooks import load_tb_data, plot_scalars, get_image, create_mol_grid

%load_ext tensorboard

In [ ]:
!pwd

set working directory

In [2]:
import os
import shutil

# ホームディレクトリの下に新しい作業ディレクトリを作成する例
reinvent_dir = os.path.expanduser("~/Documents/apps/REINVENT4")
wd = os.path.join(reinvent_dir, "tmp") # 例：~/MyJupyterProject
os.makedirs(wd, exist_ok=True) # ディレクトリが存在しない場合は作成

shutil.rmtree(os.path.join(wd, "R4_notebooks_output"), ignore_errors=True) # 以前のディレクトリを削除 (もしあれば)
wd_new = os.path.join(wd, "R4_notebooks_output")
os.makedirs(wd_new, exist_ok=True)
os.chdir(wd_new)
print(os.getcwd())

In [3]:
wd_new

set parameter file path

In [4]:
reinvent = "/Users/keetane/opt/anaconda3/envs/r4/bin/reinvent"
priors_dir = os.path.join(reinvent_dir, "priors")
Reinvent = os.path.join(priors_dir, "reinvent.prior")
lib = os.path.join(priors_dir, "libinvent.prior")
link = os.path.join(priors_dir, "linkinvent.prior")
mol2mol_high = os.path.join(priors_dir, "mol2mol_high_similarity.prior")
mol2mol_med = os.path.join(priors_dir, "mol2mol_medium_similarity.prior")
mol2mol_mmp = os.path.join(priors_dir, "mol2mol_mmp.prior")
mol2mol_scaffold_generic= os.path.join(priors_dir, "mol2mol_scaffold_generic.prior")
mol2mol_scaffold = os.path.join(priors_dir, "mol2mol_scaffold.prior")
mol2mol_similarity = os.path.join(priors_dir, "mol2mol_similarity.prior")
pubchem = os.path.join(priors_dir, "pubchem_ecfp4_with_count_with_rank_reinvent4_dict_voc.prior")
Reinvent

In [5]:
!pwd

In [9]:
toml=f'''
## Reinvent: de novo sampling
model_file = {Reinvent}
output_file = 'results/sampling0404.csv'  # sampled SMILES and NLL in CSV format

num_smiles = 157  # number of SMILES to be sampled, 1 per input SMILES
unique_molecules = true  # if true remove all duplicatesd canonicalize smiles
randomize_smiles = true # if true shuffle atoms in SMILES randomly

'''
os.makedirs('toml', exist_ok=True)
os.chdir('toml')


toml_path = f"{wd_new}/toml/toml.toml"

with open(toml_path, "w") as f:
    f.write(toml)

In [15]:
toml_path

In [12]:
subprocess.call([reinvent,
                '-l',
                'sampling.log',
                toml_path,
                ])

In [28]:
subprocess.call([
    'cat',
    toml_config_filename
])

In [29]:
subprocess.call([
    reinvent,
    '-l',
    'sampling.log',
    toml_config_filename
])

In [30]:
toml=f'''
# REINVENT4 TOML input example for sampling
#


run_type = "sampling"
device = "mps"  # set torch device e.g. "cpu". For macOS, use "mps"
json_out_config = "_sampling.json"  # write this TOML to JSON


[parameters]

# Uncomment one of the comment blocks below.  Each generator needs a model
# file and possibly a SMILES file with seed structures.

## Reinvent: de novo sampling
model_file = "priors/reinvent.prior"

## LibInvent: find R-groups for the given scaffolds
#model_file = "priors/libinvent.prior"
#smiles_file = "configs/toml/scaffolds.smi"  # 1 scaffold per line with attachment points

## LinkInvent: find a linker/scaffold to link two fragments
#model_file = "priors/linkinvent.prior"
#smiles_file = "configs/toml/warheads.smi"  # 2 warheads per line separated with '|'

## Mol2Mol: find molecules similar to the provided molecules
#model_file = "priors/mol2mol_medium_similarity.prior"
#smiles_file = "mol2mol.smi"  # 1 compound per line
#sample_strategy = "beamsearch"  # multinomial or beamsearch (deterministic)
#temperature = 1.0 # temperature in multinomial sampling
#tb_logdir = "tb_logs"  # name of the TensorBoard logging directory

output_file = 'results/sampling.csv'  # sampled SMILES and NLL in CSV format

num_smiles = 157  # number of SMILES to be sampled, 1 per input SMILES
unique_molecules = true  # if true remove all duplicatesd canonicalize smiles
randomize_smiles = true # if true shuffle atoms in SMILES randomly

'''

toml_config_filename = "sampling.toml"

with open(toml_config_filename, "w") as tf:
    tf.write(toml)
!cat {toml_config_filename}

In [34]:
!pwd

In [32]:
subprocess.call([
    reinvent,
    '-l',
    'TL.log',
    'sampling.toml',
    ])


In [33]:
!pwd

In [49]:
subprocess.call([
    "/Users/keetane/opt/anaconda3/envs/r4/bin/reinvent",
    "-l",
    "TL.log",
    "/Users/keetane/Documents/cheminfo_tutorial_20241028/data/genai/transfer_learning.toml"
    ])

# Reinvent4を使いEGFR kinase阻害剤様の構造生成モデルを作成する

転移学習とSamplingを利用してEGFR kinase阻害剤様の構造生成モデルを作成しましょう。

注）reinvent4は以下のようにCLIで利用しますが、本ハンズオンではjupyter上で実行するためsubprocessを利用します。

```bash
$ reinvent -l log.txt config.toml
```

## 転移学習に利用するEGFR kinase阻害剤の構造を確認する

rdkitを利用して構造を描画し確認します。

In [1]:
from rdkit import Chem
from rdkit.Chem import Draw
import pandas as pd
df = pd.read_csv('../data/genai/ChEMBL_EGFR.csv')
df.head(2)
Draw.MolsToGridImage([Chem.MolFromSmiles(smi) for smi in df.smiles][:20], molsPerRow=5)

## 設定ファイル(toml)を編集する

../data/genai/ 以下にTomlファイルがありますので、各自の環境に応じて編集をしてください。
以下に今回の転移学習で利用するファイル(transfer_learning.toml)を出力しますが、書き換える必要のある変数は

- device: cudaの場合はcuda:0, MacOS(M1,M2,M3)のgpuの場合はmps, gpuを利用しない場合はcpuを指定してください
- input_model_file, smiles_file, output_model_file, validation_smiles_fileは書き換える必要があります
- 今回はnum_epocを300にしていますがCPUのみの場合は学習に時間がかかるかもしれません。その場合は100に変更してください

In [2]:
!cat ../data/genai/transfer_learning.toml

## 転移学習を実施
reinventコマンドは各自の環境に依存しますので設定してください

注）仮想環境のpathを調べたい場合は以下のコマンドを実行してください

```bash
$ % conda info --envs
```

In [3]:
import subprocess

In [13]:
%ls /Users/keetane/Documents/cheminfo_tutorial_20241028/data/genai

In [5]:
subprocess.call(
    ["/Users/keetane/opt/anaconda3/envs/r4/bin/reinvent", "-l", "TL.log", "../data/genai/transfer_learning.toml"]
)

## tensroboardでログを確認
tensorboardはReinvent4と同じ環境下にインストールされているので、以下のコマンドを実行するとwebブラウザから転移学習の結果を確認することができます。

In [6]:
!/home/iwatobipen/miniforge3/envs/reinvent4/bin/tensorboard --logdir tb_TL/

##　訓練前後のモデルでサンプリング（構造生成）

ここでは、転移学習前後の生成モデルから構造生成を行い、EGFR阻害剤様の構造が出力されているか確認します。
転移学習と同様にtomlファイルを設定する必要があります。

sampling.tomlは転移学習前の生成モデルからのサンプリングで、TL_sampling.tomlは転移学習後の生成モデルからのサンプリング用の設定ファイルです。
編集が必要な箇所はdevice,model_fileです。


In [7]:
!cat ../data/genai/sampling.toml

In [5]:
#訓練前のモデル
subprocess.call(
    ["/Users/keetane/opt/anaconda3/envs/r4/bin/reinvent", "-l", "sampling.log", "../data/genai/sampling.toml"]
)

In [9]:
!cat ../data/genai/TL_sampling.toml

In [7]:
#訓練後のモデル
subprocess.call(
    ["/Users/keetane/opt/anaconda3/envs/r4/bin/reinvent", "-l", "TL_sampling.log", "../data/genai/TL_sampling.toml"]
)

## 構造を確認する

転移学習前後で生成モデルの出力する構造を確認します。転移学習後の生成モデルからはキナゾリン骨格が多く生成されていることを確認してください。

In [8]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import PandasTools

In [9]:
df1 = pd.read_csv('sampling.csv')
print(df1.shape)
df1.tail(2)

In [10]:
PandasTools.AddMoleculeColumnToFrame(df1, smilesCol='SMILES')
Draw.MolsToGridImage(df1.ROMol[:20], molsPerRow=5)

In [11]:
df2= pd.read_csv('TL_sampling.csv')
print(df2.shape)
df2.tail(2)

In [12]:
PandasTools.AddMoleculeColumnToFrame(df2, smilesCol='SMILES')
Draw.MolsToGridImage(df2.ROMol[:20], molsPerRow=5)

In [16]:
PandasTools.WriteSDF(df2, 'TL_sampling.sdf', molColName='ROMol', properties=['SMILES'])